# Colab CPU vs GPU baseline

Same sklearn MLP and PyTorch FFNN as the local project. Runtime must be **GPU**.
Upload the `python/` folder (backends, datasets, metrics) next to this notebook, or clone the repo.

In [ ]:
import os, sys, subprocess

def find_python_root():
    here = os.getcwd()
    for candidate in [here, os.path.join(here, 'python'), '/content/python', '/content']:
        if os.path.isfile(os.path.join(candidate, 'backends', 'gpu_baseline.py')):
            return candidate
        nested = os.path.join(candidate, 'benchmark-neuromorphic', 'python')
        if os.path.isfile(os.path.join(nested, 'backends', 'gpu_baseline.py')):
            return nested
    raise FileNotFoundError('Upload the project python/ folder (must contain backends/gpu_baseline.py)')

ROOT = find_python_root()
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('python root:', ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'pandas', 'numpy', 'psutil'])

In [ ]:
import os, sys, torch

if 'ROOT' in globals() and ROOT in sys.path:
    sys.path.remove(ROOT)
if 'ROOT' in globals():
    sys.path.insert(0, ROOT)

assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'
from colab.run_colab_cpu_gpu import assert_cuda
print('CUDA:', assert_cuda())

In [ ]:
from colab.run_colab_cpu_gpu import run_pair
import json, pandas as pd

cicids = os.environ.get('CICIDS_PATH')
unsw = os.environ.get('UNSW_NB15_PATH')
for path in ['/content/cicids.csv', os.path.join(ROOT, '..', 'storage', 'datasets', 'cicids.csv')]:
    if not cicids and os.path.isfile(path):
        cicids = path
for path in ['/content/unsw_nb15.csv', os.path.join(ROOT, '..', 'storage', 'datasets', 'unsw_nb15.csv')]:
    if not unsw and os.path.isfile(path):
        unsw = path

results = []
results.extend(run_pair('cicids', cicids))
results.extend(run_pair('unsw_nb15', unsw))

out = os.path.join(ROOT, 'colab', 'colab_results.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w', encoding='utf-8') as fh:
    json.dump({'results': results}, fh, indent=2)

cols = ['dataset', 'architecture', 'backend', 'f1_score', 'latency_ms', 'throughput_ops_per_sec',
        'energy_joules_per_op', 'gpu_utilization', 'cpu_utilization', 'n_test']
table = []
for row in results:
    table.append({
        'dataset': row['dataset'],
        'architecture': row['architecture'],
        'backend': row.get('backend'),
        'f1_score': row.get('f1_score'),
        'latency_ms': row.get('latency_ms'),
        'throughput_ops_per_sec': row.get('throughput_ops_per_sec'),
        'energy_joules_per_op': row.get('energy_joules_per_op'),
        'gpu_utilization': row.get('gpu_utilization'),
        'cpu_utilization': row.get('cpu_utilization'),
        'n_test': row.get('split', {}).get('n_test'),
    })
display(pd.DataFrame(table))
print('Wrote', out)